# Continuous segment embeddings versus discrete units

In [ ]:
from __future__ import annotations

import sys
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

for _cand in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_cand / "crossfarm_nested.py").exists():
        sys.path.insert(0, str(_cand))
        break
import crossfarm_nested as cn
import layer3_reliability as rel

UNITS_DIR = cn.OUT / "14n_layer2_crossfarm_nested"
UNITS_L3 = cn.OUT / "19n_layer3_reliability_analysis"
CNN_DIR = cn.OUT / "cnn_baseline"
OUT_DIR = cn.OUT / "14c_layer2_continuous_embeddings"
OUT_DIR.mkdir(parents=True, exist_ok=True)
TARGETS = ["is_nonvocal", "is_distress_related", "is_resource_related", "proxy_functional_v1"]
COVERAGES = [1.0, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50]
RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

clips = cn.load_clip_table()
unit_settings = {int(k): v for k, v in json.load(open(UNITS_DIR / "crossfarm_manifest.json"))["fold_settings"].items()}
fold_settings = {h: {"segmentation": s["segmentation"], "k": None} for h, s in unit_settings.items()}
fold_settings

In [ ]:
pred_df, selection_df, fold_info = cn.run_lofo(clips, fold_settings, nested_inner=True,
                                               representation="continuous")
metrics_df, perfarm_df = cn.summarise(pred_df)

inner_df, inner_check = cn.inner_predictions(clips, fold_settings, pred_df, representation="continuous")
inner_check = inner_check.merge(selection_df.astype({"heldout_farm": str}),
                                on=["heldout_farm", "target_name", "bundle_name"])
inner_check["abs_diff"] = (inner_check["inner_macro_f1"] - inner_check["inner_macro_f1_recomputed"]).abs()
assert inner_check["abs_diff"].max() < 1e-9, inner_check
print("inner predictions:", len(inner_df), "rows; selection scores reproduced (max diff",
      inner_check["abs_diff"].max(), ")")

In [ ]:
co_o = rel.standardise_predictions(pred_df, "target_name", "heldout_farm")
co_i = rel.standardise_predictions(inner_df, "target_name", "heldout_farm", "inner_val_farm")
un_o = rel.standardise_predictions(pd.read_csv(UNITS_DIR / "crossfarm_fold_predictions.csv.gz"),
                                   "target_name", "heldout_farm")
un_i = rel.standardise_predictions(pd.read_csv(UNITS_L3 / "framework_inner_predictions.csv.gz"),
                                   "target_name", "heldout_farm", "inner_val_farm")
cnn_o = rel.standardise_predictions(pd.read_csv(CNN_DIR / "cnn_lofo_predictions.csv.gz"), "target", "held_farm")
cnn_i = rel.standardise_predictions(pd.read_csv(CNN_DIR / "cnn_lofo_inner_predictions.csv.gz"),
                                    "target", "held_farm", "inner_val_farm")

results = {}
for name, (o, i) in {"discrete_units": (un_o, un_i), "continuous_embeddings": (co_o, co_i),
                     "cnn_logmel": (cnn_o, cnn_i)}.items():
    results[name] = {"outer": o, "l3": rel.summarise_model(o, i, TARGETS, COVERAGES)}
co_store, co_calib, co_select, co_curves, co_folds, co_pooled = results["continuous_embeddings"]["l3"]

In [ ]:
floor = metrics_df[metrics_df.model_name == "dummy_most_frequent"].set_index("target_name")["macro_f1"]


def macro(d):
    return f1_score(d["y"], rel.prediction_matrix(d)[0].argmax(1), average="macro", zero_division=0)


rows, perfarm_rows = [], []
for name, res in results.items():
    o = res["outer"]
    calib, select = res["l3"][1].set_index("target"), res["l3"][2].set_index("target")
    for t in TARGETS:
        ot = o[o.target == t]
        rows.append({"representation": name, "target": t, "lofo_macro_f1": macro(ot),
                     "majority_floor_macro_f1": floor[t],
                     "ece_raw": calib.loc[t, "ece_raw"], "ece_nested_calibrated": calib.loc[t, "ece_selected"],
                     "calibration_method": calib.loc[t, "selected_method"],
                     "selective_score": select.loc[t, "score_version"],
                     "retained_macro_f1_80cov": select.loc[t, "retained_macro_f1_80cov"],
                     "selective_gain": select.loc[t, "selective_gain"]})
        for held, oh in ot.groupby("held"):
            perfarm_rows.append({"representation": name, "target": t, "heldout_farm": held,
                                 "n_clips": len(oh), "macro_f1": macro(oh)})
comparison_df = pd.DataFrame(rows)
perfarm_comparison_df = pd.DataFrame(perfarm_rows)
wide = comparison_df.pivot(index="target", columns="representation",
                           values=["lofo_macro_f1", "retained_macro_f1_80cov", "ece_raw"]).loc[TARGETS]
wide

In [ ]:
pred_df.to_csv(OUT_DIR / "continuous_fold_predictions.csv.gz", index=False, compression="gzip")
selection_df.to_csv(OUT_DIR / "continuous_bundle_selection_scores.csv", index=False)
metrics_df.to_csv(OUT_DIR / "continuous_metrics_summary.csv", index=False)
perfarm_df.to_csv(OUT_DIR / "continuous_bestbundle_perfarm_metrics.csv", index=False)
fold_info.to_csv(OUT_DIR / "continuous_fold_settings.csv", index=False)
inner_df.to_csv(OUT_DIR / "continuous_inner_predictions.csv.gz", index=False, compression="gzip")
co_calib.to_csv(OUT_DIR / "continuous_calibration_summary.csv", index=False)
co_folds.to_csv(OUT_DIR / "continuous_calibration_selection_by_fold.csv", index=False)
co_select.to_csv(OUT_DIR / "continuous_selective_prediction_summary.csv", index=False)
co_curves.to_csv(OUT_DIR / "continuous_selective_coverage_curves.csv", index=False)
comparison_df.to_csv(OUT_DIR / "representation_comparison.csv", index=False)
perfarm_comparison_df.to_csv(OUT_DIR / "representation_comparison_perfarm.csv", index=False)

selected = (pred_df.groupby(["heldout_farm", "target_name"])["bundle_name"].first().reset_index())
selected.to_csv(OUT_DIR / "continuous_selected_bundles.csv", index=False)
with open(OUT_DIR / "continuous_manifest.json", "w", encoding="utf-8") as f:
    json.dump({"build_ts_utc": RUN_TS, "fold_settings": {str(k): v for k, v in fold_settings.items()},
               "families": cn.CONTINUOUS_FAMILIES, "context_variants": cn.CONTEXT_VARIANTS,
               "pca_dim": cn.PCA_DIM, "classifier": "logistic regression, class-balanced (as for units)",
               "inner_loop": "features rebuilt from the inner training farm",
               "layer3": "calibrator and score chosen per held-out farm from out-of-farm predictions "
                         "inside its training partition"}, f, indent=2)

In [ ]:
pd.set_option("display.width", 250)
display(selected)
display(comparison_df.round(3))
display(perfarm_comparison_df.pivot_table(index=["target", "heldout_farm"], columns="representation",
                                          values="macro_f1").round(3))
display(co_folds.round(3))